# 💳 Credit Card Transaction Analysis

**Dataset:** Simulated US Credit Card Transactions (Kaggle)  
**Goal:** Explore spending patterns across job groups, categories, and days of the week to identify high-value customer segments.

---
### 📁 Files Required
| File | Description |
|---|---|
| `credit_card_transactions.csv` | Main transaction dataset |
| `grouped_jobs.csv` | Job name normalization mapping |

---
### 📌 Table of Contents
1. [Data Loading & Preprocessing](#1-data-loading--preprocessing)
2. [Feature Engineering](#2-feature-engineering)
3. [Job Group Analysis](#3-job-group-analysis)
4. [Weekly Pattern by Job](#4-weekly-pattern-by-job)
5. [Deep Dive: Top Job Groups](#5-deep-dive-top-job-groups)
6. [Regional Distribution](#6-regional-distribution)


## 1. Data Loading & Preprocessing

### 1.1 Import Libraries

In [ ]:
import pandas as pd 
import numpy as np
import time
from PIL import Image
import altair as alt
import seaborn as sns
import matplotlib.pyplot as plt 

### 1.2 Load Data

Selected columns only to reduce memory usage.  
`grouped_jobs.csv` maps raw job titles into grouped categories (e.g., 'Software Engineer' → 'engineer').


In [ ]:
columns = ['trans_date_trans_time', 'category', 'amt', 'gender', 'state', 'dob', 'job', 'first', 'last']
df = pd.read_csv('credit_card_transactions.csv', usecols=columns)

# Normalize job names using external mapping
job_mapping = pd.read_csv('grouped_jobs.csv')
job_mapping_dict = dict(zip(job_mapping['job'], job_mapping['rename_job']))
df['job'] = df['job'].replace(job_mapping_dict)

# Merge pos/net variants into single category labels
df['category'] = df['category'].replace({
    'grocery_net': 'grocery', 'grocery_pos': 'grocery',
    'shopping_net': 'shopping', 'shopping_pos': 'shopping',
    'misc_net': 'misc', 'misc_pos': 'misc'
})

df.info()

### 1.3 Region Mapping

US states are grouped into geographic belts to enable regional comparison.


In [ ]:
# 벨트 기준 매핑 딕셔너리
state_to_region = {
    # Tech Belt
    'CA': 'Tech Belt', 'WA': 'Tech Belt', 'OR': 'Tech Belt',
    'MA': 'Tech Belt', 'CO': 'Tech Belt', 'UT': 'Tech Belt',

    # Rust Belt
    'IL': 'Rust Belt', 'IN': 'Rust Belt', 'MI': 'Rust Belt',
    'OH': 'Rust Belt', 'PA': 'Rust Belt', 'WI': 'Rust Belt',
    'MO': 'Rust Belt', 'IA': 'Rust Belt', 'MN': 'Rust Belt',

    # Sun Belt
    'TX': 'Sun Belt', 'FL': 'Sun Belt', 'AZ': 'Sun Belt',
    'NV': 'Sun Belt', 'GA': 'Sun Belt', 'NC': 'Sun Belt',
    'SC': 'Sun Belt', 'NM': 'Sun Belt', 'TN': 'Sun Belt',

    # Bible Belt
    'AL': 'Bible Belt', 'MS': 'Bible Belt', 'AR': 'Bible Belt',
    'KY': 'Bible Belt', 'OK': 'Bible Belt', 'LA': 'Bible Belt',
    'WV': 'Bible Belt',

    # Mountain States
    'MT': 'Mountain States', 'ID': 'Mountain States', 'WY': 'Mountain States',
    'ND': 'Mountain States', 'SD': 'Mountain States', 'NE': 'Mountain States',
    'KS': 'Mountain States',

    # New England
    'CT': 'New England', 'RI': 'New England', 'VT': 'New England',
    'NH': 'New England', 'ME': 'New England',

    # Middle Atlantic
    'NY': 'Middle Atlantic', 'NJ': 'Middle Atlantic', 'DE': 'Middle Atlantic',
    'MD': 'Middle Atlantic', 'DC': 'Middle Atlantic', 'VA': 'Middle Atlantic',

    # Pacific Islands
    'AK': 'Pacific Islands', 'HI': 'Pacific Islands'
}
# region_group 컬럼 생성
df['region_group'] = df['state'].map(state_to_region)


## 2. Feature Engineering

New columns derived from `trans_date_trans_time` and `dob`:
- `month` — transaction month (1–12)
- `weekday` — day name (Monday–Sunday)
- `week_type` — Weekday / Weekend
- `user_id` — unique customer identifier (name + dob hashing)


In [ ]:
# Parse datetime columns
df['trans_date_trans_time'] = pd.to_datetime(df['trans_date_trans_time'])
df['dob'] = pd.to_datetime(df['dob'])

# Extract time features
df.insert(1, 'month', df['trans_date_trans_time'].dt.month)
df.insert(2, 'weekday', df['trans_date_trans_time'].dt.day_name())
df['week_type'] = df['trans_date_trans_time'].dt.weekday.apply(lambda x: 'Weekend' if x >= 5 else 'Weekday')

# Assign unique user IDs (name + date of birth)
df['user_key'] = df['first'] + '_' + df['last'] + '_' + df['dob'].astype(str)
df['user_id'] = df['user_key'].astype('category').cat.codes

print(f"Total records: {len(df):,}")
print(f"Unique users:  {df['user_id'].nunique():,}")
df.head(5)

## 3. Job Group Analysis

### 3.1 Build Job Ranking Table

Aggregate by job group: transaction count, total spending, unique user count, and per-user/per-purchase metrics.


In [ ]:
# Aggregate by job
job_rank = df.groupby('job').size().reset_index(name='count')
amt_sum  = df.groupby('job')['amt'].sum().reset_index()
job_rank = job_rank.merge(amt_sum, on='job')

# Dual ranking: by user count and by total spending
job_rank.sort_values(by='count', ascending=False, inplace=True)
job_rank['count_rank'] = range(1, len(job_rank) + 1)

job_rank.sort_values(by='amt', ascending=False, inplace=True)
job_rank['amt_rank'] = range(1, len(job_rank) + 1)

# Filter top 20 by user count
job_count_filtered = job_rank[job_rank['count_rank'] <= 20].sort_values('count_rank').copy()

# Add unique user count and per-user metrics
job_user_counts = df.groupby('job')['user_id'].nunique().reset_index(name='user_count')
job_count_filtered = job_count_filtered.merge(job_user_counts, on='job', how='left')

job_count_filtered['mean_count']     = (job_count_filtered['count'] / job_count_filtered['user_count']).round()
job_count_filtered['mean_amt']       = (job_count_filtered['amt']   / job_count_filtered['user_count']).round(2)
job_count_filtered['purchase_value'] = (job_count_filtered['amt']   / job_count_filtered['count']).round(2)

job_count_filtered

### 3.2 Top 20 Jobs — User Count Distribution

In [ ]:
# Calculate Job Ratio
job_counts = job_count_filtered[['job', 'user_count']].sort_values(by='user_count', ascending=False)

# Color
colors = sns.color_palette('crest', n_colors=len(job_counts))

# Pie chart
plt.figure(figsize=(6, 6))
plt.pie(job_counts['user_count'], 
        labels=job_counts.job, 
        autopct='%1.1f%%', 
        startangle=90, 
        colors=colors)
plt.title('Job Ratio')
plt.axis('equal')
plt.savefig('Job Ratio', bbox_inches='tight')
plt.show()
plt.show()

### 3.3 Top 20 Jobs — User Count Bar

In [ ]:
# 데이터 정렬
job_count_filtered.sort_values(by='user_count', ascending=False, inplace=True)

# catplot으로 그리되, 객체 반환
g = sns.catplot(
    data=job_count_filtered,
    x='user_count',
    y='job',
    kind="bar",
    palette="crest",
    height=6,
    aspect=1.2
)

# 축 객체 추출
ax = g.ax

# 바 하나씩 순회하며 값 표시
for p in ax.patches:
    value = int(p.get_width())
    ax.text(
        p.get_width() + 5,               # x 위치 (막대 끝 + 여유)
        p.get_y() + p.get_height() / 2,  # y 위치 (막대 중앙)
        f'{value:,}',                    # 텍스트 (콤마 포함)
        va='center',
        fontsize=9
    )

# 타이틀 및 레이블
plt.title('Top 20 Jobs', fontsize=14)
plt.xlabel('User Count')
plt.ylabel('Job')

# 저장 및 출력
plt.savefig('Top 20 Jobs', bbox_inches='tight')
plt.show()

### 3.4 Market Size — Total Spending by Job

In [ ]:
# Total Spending by Job Group
job_count_filtered.sort_values(by='amt', ascending=False, inplace=True)

# Bar chart
g = sns.catplot(data=job_count_filtered, x='amt', y='job', kind="bar", hue="amt", palette="crest", height=6, aspect=1.2)

# 축 객체 추출
ax = g.ax

# 바 하나씩 순회하며 값 표시
for p in ax.patches:
    value = int(p.get_width())
    ax.text(
        p.get_width() + 5,               # x 위치 (막대 끝 + 여유)
        p.get_y() + p.get_height() / 2,  # y 위치 (막대 중앙)
        f'{value:,}',                    # 텍스트 (콤마 포함)
        va='center',
        fontsize=9
    )

plt.title('Total Spending by Job - Market Size')
plt.xlabel('Spending')
plt.ylabel('Job')
plt.savefig('Total Spending by Job_bar', bbox_inches='tight')
plt.show()

### 3.5 Total Spending vs. Spending per User

Bar = total market size (left axis) · Line = per-user spending (right axis).  
High bar + low line → large group but low individual value.  
Low bar + high line → niche group with high-value customers.


In [ ]:
job_count_filtered.sort_values(by='amt', ascending=False, inplace=True)

# Total spending & count by job
fig, ax1 = plt.subplots(figsize=(10, 6))

# 왼쪽 y축
ax1.bar(job_count_filtered['job'], job_count_filtered['amt'], color='pink')
ax1.set_ylabel('Total Spending', color='pink')
ax1.tick_params(axis='y', labelcolor='red')

# 오른쪽 y축
ax2 = ax1.twinx()
ax2.plot(job_count_filtered['job'], job_count_filtered['mean_amt'], color='blue', marker='s')
ax2.set_ylabel('Spending per user', color='blue')
ax2.tick_params(axis='y', labelcolor='blue')

plt.title('Total Spending & Spending per User by Job')
ax1.set_xticks(range(len(job_amt_filtered['job'])))
ax1.set_xticklabels(job_amt_filtered['job'], rotation=45, ha='right')
plt.savefig('Total Spending & Spending per User by Job', bbox_inches='tight')
plt.show()

### 3.6 Purchase Behavior — Frequency vs. Spend per Transaction

In [ ]:
job_count_filtered.sort_values(by='mean_count', ascending=False, inplace=True)

# Mean spending & count by job
fig, ax1 = plt.subplots(figsize=(10, 6))

# 왼쪽 y축
ax1.bar(job_count_filtered['job'], job_count_filtered['purchase_value'], color='orange')
ax1.set_ylabel('Mean Spending per Purchase', color='orange')
ax1.tick_params(axis='y', labelcolor='orange')

# 오른쪽 y축
ax2 = ax1.twinx()
ax2.plot(job_count_filtered['job'], job_count_filtered['mean_count'], color='blue', marker='s')
ax2.set_ylabel('Mean Purchase Count', color='blue')
ax2.tick_params(axis='y', labelcolor='blue')

plt.title('Mean Spending per Purchase & Mean Purchase Count by Job')
ax1.set_xticks(range(len(job_count_filtered['job'])))
ax1.set_xticklabels(job_count_filtered['job'], rotation=45, ha='right')
plt.savefig('Mean Spending per Purchase & Purchase Count by Job', bbox_inches='tight')
plt.show()

In [ ]:
job_count_filtered.sort_values(by='mean_count', ascending=False, inplace=True)

# Mean spending & count by job
fig, ax1 = plt.subplots(figsize=(10, 6))

# 왼쪽 y축
ax1.bar(job_count_filtered['job'], job_count_filtered['mean_amt'], color='orange')
ax1.set_ylabel('Per Person Total Spending', color='orange')
ax1.tick_params(axis='y', labelcolor='orange')

# 오른쪽 y축
ax2 = ax1.twinx()
ax2.plot(job_count_filtered['job'], job_count_filtered['mean_count'], color='blue', marker='s')
ax2.set_ylabel('Per Person Purchase Count', color='blue')
ax2.tick_params(axis='y', labelcolor='blue')

plt.title('Per Person Spending & Per Person Purchase by Job')
ax1.set_xticks(range(len(job_count_filtered['job'])))
ax1.set_xticklabels(job_count_filtered['job'], rotation=45, ha='right')
plt.savefig('Per Person Spending & Per Person Purchase by Job', bbox_inches='tight')
plt.show()

### 3.7 Top 20 Jobs Coverage

Check what share of total transactions and spending the top-20 job groups represent.


In [ ]:
top20_count_share = job_count_filtered['count'].sum() / df['job'].count()
top20_amt_share   = job_count_filtered['amt'].sum()   / df['amt'].sum()
print(f"Top 20 jobs — transaction share: {top20_count_share:.1%}")
print(f"Top 20 jobs — spending share:    {top20_amt_share:.1%}")

## 4. Weekly Pattern by Job

### 4.1 Setup — Filtered Dataset

Use top-20 job users for focused weekly analysis.


In [ ]:
# Build filtered dataset: only users in top-20 job groups
top20_jobs = job_count_filtered['job'].tolist()
df_filtered = df[df['job'].isin(top20_jobs)].copy()
print(f"Filtered dataset: {len(df_filtered):,} rows, {df_filtered['user_id'].nunique():,} users")
df_filtered.head(3)

### 4.2 Heatmap — Job × Day of Week

In [ ]:
weekday_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
pivot = pd.pivot_table(df_filtered, index='job', columns='weekday',
                       values='trans_date_trans_time', aggfunc='count', fill_value=0)
pivot = pivot[weekday_order]
pivot.sort_values(by='Monday', ascending=False, inplace=True)

plt.figure(figsize=(12, 8))
sns.heatmap(pivot, annot=True, fmt='.0f', cmap='YlGnBu')
plt.title('Purchase Count — Job vs Days of the Week')
plt.xlabel('Day')
plt.ylabel('Job')
plt.tight_layout()
plt.savefig('Purchase_Count_Job_vs_Weekday.png', bbox_inches='tight')
plt.show()

### 4.3 Heatmap — Category × Day of Week

In [ ]:
days = df_filtered.groupby(['weekday', 'category']).size().reset_index(name='count')
pivot2 = days.pivot(index='weekday', columns='category', values='count').fillna(0)
pivot2 = pivot2.reindex(weekday_order)

plt.figure(figsize=(12, 6))
sns.heatmap(pivot2, annot=True, fmt='.0f', cmap='YlGnBu')
plt.title('Purchase Count — Category vs Days of the Week')
plt.xlabel('Category')
plt.ylabel('Weekday')
plt.tight_layout()
plt.savefig('Category_vs_Days_Heatmap.png', bbox_inches='tight')
plt.show()

### 4.4 Monday Focus — Job × Category Composition

Monday shows distinct patterns vs. other weekdays. Visualise job-wise category mix.

In [ ]:
# Monday top 20
plt.figure(figsize=(12, 8))
sns.barplot(data=monday_cat, x='count', y='job', hue='category', dodge=False, palette='Set2')
plt.title('Monday: Job-Category')
plt.xlabel('Count')
plt.ylabel('Job')
plt.legend(title='Category')
plt.tight_layout()
plt.savefig('Monday: Job-Category', bbox_inches='tight')
plt.show()

In [ ]:
# stacked bar chart
# 1. 직업-카테고리별 총 구매 건수
monday_cat_pct = monday_cat.copy()

# 2. 직업별 총 구매 건수
job_total = monday_cat_pct.groupby('job')['count'].sum().reset_index(name='total')

# 3. 병합 후 퍼센트 계산
monday_cat_pct = monday_cat_pct.merge(job_total, on='job')
monday_cat_pct['percent'] = (monday_cat_pct['count'] / monday_cat_pct['total']) * 100
# 피벗: 직업별로 각 카테고리의 퍼센트
pivot_pct = monday_cat_pct.pivot(index='job', columns='category', values='percent').fillna(0)

# 시각화를 위해 직업 정렬 (총 소비 많은 순)
pivot_pct = pivot_pct.loc[pivot_pct.sum(axis=1).sort_values(ascending=False).index]

In [ ]:
# 색상 팔레트 지정
colors = plt.cm.tab20.colors  # 최대 20개의 카테고리

# 그래프 크기
plt.figure(figsize=(16, 10))

bottom = [0] * len(pivot_pct)
x = range(len(pivot_pct))

# 카테고리 순서대로 누적해서 그리기
for i, category in enumerate(pivot_pct.columns):
    plt.bar(x, pivot_pct[category], bottom=bottom, label=category, color=colors[i % len(colors)])
    bottom = [a + b for a, b in zip(bottom, pivot_pct[category])]

# x축 설정
plt.xticks(ticks=x, labels=pivot_pct.index, rotation=90)
plt.ylabel('Percent (%)')
plt.title('Monday: Job-wise Category Composition (100% Stacked)')
plt.ylim(0, 100)
plt.legend(title='Category', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()

# 저장
plt.savefig('Monday_Jobwise_Category_Percent.png', bbox_inches='tight')
plt.show()

### 4.5 Monthly Purchase Trend by Job (2019)

In [ ]:
# 2019년 데이터만 필터링
df_2019 = df_filtered[df_filtered['trans_date_trans_time'].dt.year == 2019].copy()

# 월 컬럼 생성
df_2019['month'] = df_2019['trans_date_trans_time'].dt.month

# 월별, 직업별 구매 건수 집계
job_month_count = df_2019.groupby(['job', 'month']).size().reset_index(name='count')

# 시각화
plt.figure(figsize=(14, 8))
sns.lineplot(data=job_month_count, x='month', y='count', hue='job', marker='o', palette='tab10')

plt.title('Monthly Purchase Pattern by Job (2019)', fontsize=16)
plt.xlabel('Month')
plt.ylabel('Purchase Count')
plt.xticks(range(1, 13))
plt.legend(title='Job', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig('Monthly_Purchase_Pattern_by_Job_2019.png', bbox_inches='tight')
plt.show()

## 5. Deep Dive: Top Job Groups

Drill into three representative job groups — **Engineer**, **Environment**, **Public Sector** —  
to compare weekly spending rhythm and category preferences.


### 5.1 Engineer

In [ ]:
engineer = df[df['job'] == 'engineer'].copy()
n_eng = engineer['user_id'].nunique()
print(f"Engineer users: {n_eng:,}")

eng_day = engineer.groupby('weekday').size().reset_index(name='count')
eng_day_amt = engineer.groupby('weekday')['amt'].sum().reset_index(name='amt')
eng_day = eng_day.merge(eng_day_amt, on='weekday')

weekday_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
eng_day['weekday'] = pd.Categorical(eng_day['weekday'], categories=weekday_order, ordered=True)
eng_day = eng_day.sort_values('weekday')
eng_day['user_purchase'] = (eng_day['count'] / n_eng).round()
eng_day['user_amt']      = (eng_day['amt']   / n_eng).round(2)
eng_day

In [ ]:
# 레이아웃 설정
fig, (ax1, ax3) = plt.subplots(1, 2, figsize=(16, 6), sharex=True)

# -------- 왼쪽 그래프: Total Spending & Purchase Count --------
# 왼쪽 y축
ax1.bar(eng_day['weekday'], eng_day['amt'], color='orange')
ax1.set_ylabel('Total Spending by Engineers', color='orange')
ax1.tick_params(axis='y', labelcolor='orange')

# 오른쪽 y축
ax2 = ax1.twinx()
ax2.plot(eng_day['weekday'], eng_day['count'], color='blue', marker='s')
ax2.set_ylabel('Purchase Count', color='blue')
ax2.tick_params(axis='y', labelcolor='blue')

# 타이틀 및 x축
ax1.set_title('Total Spending & Purchase Count of Engineers')
ax1.set_xticks(range(len(eng_day['weekday'])))
ax1.set_xticklabels(eng_day['weekday'], rotation=45, ha='right')

# -------- 오른쪽 그래프: Mean Spending & Purchase Count per User --------
# 왼쪽 y축
ax3.bar(eng_day['weekday'], eng_day['user_amt'], color='pink')
ax3.set_ylabel('Mean Spending by Engineers', color='pink')
ax3.tick_params(axis='y', labelcolor='pink')

# 오른쪽 y축
ax4 = ax3.twinx()
ax4.plot(eng_day['weekday'], eng_day['user_purchase'], color='blue', marker='s')
ax4.set_ylabel('Purchase Count per User', color='blue')
ax4.tick_params(axis='y', labelcolor='blue')

# 타이틀 및 x축
ax3.set_title('Mean Spending & Purchase Count per User - Engineer')
ax3.set_xticks(range(len(eng_day['weekday'])))
ax3.set_xticklabels(eng_day['weekday'], rotation=45, ha='right')

# 레이아웃 저장 및 출력
plt.tight_layout()
plt.savefig('Engineer_Spending_Comparison_Weekly.png', bbox_inches='tight')
plt.show()

In [ ]:
# 요일 그룹 정의
weekend = ['Monday', 'Saturday', 'Sunday']
weekday = ['Tuesday', 'Wednesday', 'Thursday', 'Friday']

# 필터링
weekend_data = df[(df['job'] == 'engineer') & (df['weekday'].isin(weekend))]
weekday_data = df[(df['job'] == 'engineer') & (df['weekday'].isin(weekday))]

# category별 count
weekend_counts = weekend_data['category'].value_counts().reset_index()
weekend_counts.columns = ['category', 'weekend_count']

weekday_counts = weekday_data['category'].value_counts().reset_index()
weekday_counts.columns = ['category', 'weekday_count']

# 병합해서 category 순서 일치
merged = pd.merge(weekend_counts, weekday_counts, on='category', how='outer').fillna(0)
merged = merged.sort_values(by='weekend_count', ascending=False)  # 정렬 기준은 weekend

# 시각화
plt.figure(figsize=(12, 6))

# 막대 그래프: weekend
sns.barplot(data=merged, x='category', y='weekend_count', color='skyblue', label='Mon/Sat/Sun (Bar)')

# 선 그래프: weekday
plt.plot(merged['category'], merged['weekday_count'], color='orange', marker='o', label='Tue–Fri (Line)')

# 라벨 및 꾸미기
plt.title("Engineer: Category Purchases – Mon/Sat/Sun(Bar) vs Tue-Fri(Line)", fontsize=14)
plt.xlabel("Category")
plt.ylabel("Purchase Count")
plt.xticks(rotation=45, ha='right')
plt.legend()
plt.tight_layout()
plt.savefig("Engineer_Category_Comparison_Line_Bar.png", bbox_inches='tight')
plt.show()

### 5.2 Environment

In [ ]:
environment = df[df['job'] == 'environment'].copy()
n_env = environment['user_id'].nunique()
print(f"Environment users: {n_env:,}")

env_day = environment.groupby('weekday').size().reset_index(name='count')
env_day_amt = environment.groupby('weekday')['amt'].sum().reset_index(name='amt')
env_day = env_day.merge(env_day_amt, on='weekday')
env_day['weekday'] = pd.Categorical(env_day['weekday'], categories=weekday_order, ordered=True)
env_day = env_day.sort_values('weekday')
env_day['user_purchase'] = (env_day['count'] / n_env).round()
env_day['user_amt']      = (env_day['amt']   / n_env).round(2)
env_day

In [ ]:
# 레이아웃 설정
fig, (ax1, ax3) = plt.subplots(1, 2, figsize=(16, 6), sharex=True)

# -------- 왼쪽 그래프: Total Spending & Purchase Count --------
# 왼쪽 y축 (막대)
ax1.bar(env_day['weekday'], env_day['amt'], color='orange')
ax1.set_ylabel('Total Spending by Environment Job', color='orange')
ax1.tick_params(axis='y', labelcolor='orange')

# 오른쪽 y축 (선)
ax2 = ax1.twinx()
ax2.plot(env_day['weekday'], env_day['count'], color='blue', marker='s')
ax2.set_ylabel('Purchase Count', color='blue')
ax2.tick_params(axis='y', labelcolor='blue')

# 타이틀 및 x축
ax1.set_title('Total Spending & Purchase Count of Environment Job')
ax1.set_xticks(range(len(env_day['weekday'])))
ax1.set_xticklabels(env_day['weekday'], rotation=45, ha='right')

# -------- 오른쪽 그래프: Mean Spending & Purchase Count per User --------
# 왼쪽 y축 (막대)
ax3.bar(env_day['weekday'], env_day['user_amt'], color='pink')
ax3.set_ylabel('Mean Spending per User', color='pink')
ax3.tick_params(axis='y', labelcolor='pink')

# 오른쪽 y축 (선)
ax4 = ax3.twinx()
ax4.plot(env_day['weekday'], env_day['user_purchase'], color='blue', marker='s')
ax4.set_ylabel('Purchase Count per User', color='blue')
ax4.tick_params(axis='y', labelcolor='blue')

# 타이틀 및 x축
ax3.set_title('Mean Spending & Purchase Count per User - Environment job')
ax3.set_xticks(range(len(env_day['weekday'])))
ax3.set_xticklabels(env_day['weekday'], rotation=45, ha='right')

# 저장 및 출력
plt.tight_layout()
plt.savefig('Environment_Spending_Comparison_Weekly.png', bbox_inches='tight')
plt.show()

In [ ]:
# 요일 그룹 정의
weekend = ['Monday', 'Saturday', 'Sunday']
weekday = ['Tuesday', 'Wednesday', 'Thursday', 'Friday']

# 필터링 (직업명: Environment Job)
weekend_data = df[(df['job'] == 'environment') & (df['weekday'].isin(weekend))]
weekday_data = df[(df['job'] == 'environment') & (df['weekday'].isin(weekday))]

# category별 count
weekend_counts = weekend_data['category'].value_counts().reset_index()
weekend_counts.columns = ['category', 'weekend_count']

weekday_counts = weekday_data['category'].value_counts().reset_index()
weekday_counts.columns = ['category', 'weekday_count']

# 병합해서 category 순서 일치
merged = pd.merge(weekend_counts, weekday_counts, on='category', how='outer').fillna(0)
merged = merged.sort_values(by='weekend_count', ascending=False)

# 시각화
plt.figure(figsize=(12, 6))

# 막대 그래프: Mon/Sat/Sun
sns.barplot(data=merged, x='category', y='weekend_count', color='skyblue', label='Mon/Sat/Sun (Bar)')

# 선 그래프: Tue–Fri
plt.plot(merged['category'], merged['weekday_count'], color='orange', marker='o', label='Tue–Fri (Line)')

# 라벨 및 꾸미기
plt.title("Environment Job: Category Purchases – Mon/Sat/Sun(Bar) vs Tue–Fri(Line)", fontsize=14)
plt.xlabel("Category")
plt.ylabel("Purchase Count")
plt.xticks(rotation=45, ha='right')
plt.legend()
plt.tight_layout()
plt.savefig("Environment_Job_Category_Comparison_Line_Bar.png", bbox_inches='tight')
plt.show()


### 5.3 Public Sector

In [ ]:
public = df[df['job'] == 'public sector'].copy()
n_pub = public['user_id'].nunique()
print(f"Public sector users: {n_pub:,}")

pub_day = public.groupby('weekday').size().reset_index(name='count')
pub_day_amt = public.groupby('weekday')['amt'].sum().reset_index(name='amt')
pub_day = pub_day.merge(pub_day_amt, on='weekday')
pub_day['weekday'] = pd.Categorical(pub_day['weekday'], categories=weekday_order, ordered=True)
pub_day = pub_day.sort_values('weekday')
pub_day['user_purchase'] = (pub_day['count'] / n_pub).round()
pub_day['user_amt']      = (pub_day['amt']   / n_pub).round(2)
pub_day

In [ ]:
# 레이아웃 설정
fig, (ax1, ax3) = plt.subplots(1, 2, figsize=(16, 6), sharex=True)

# -------- 왼쪽 그래프: Total Spending & Purchase Count --------
# 왼쪽 y축 (막대)
ax1.bar(pub_day['weekday'], pub_day['amt'], color='orange')
ax1.set_ylabel('Total Spending by Public Sector', color='orange')
ax1.tick_params(axis='y', labelcolor='orange')

# 오른쪽 y축 (선)
ax2 = ax1.twinx()
ax2.plot(pub_day['weekday'], pub_day['count'], color='blue', marker='s')
ax2.set_ylabel('Purchase Count', color='blue')
ax2.tick_params(axis='y', labelcolor='blue')

# 타이틀 및 x축
ax1.set_title('Total Spending & Purchase Count of Public Sector')
ax1.set_xticks(range(len(pub_day['weekday'])))
ax1.set_xticklabels(pub_day['weekday'], rotation=45, ha='right')

# -------- 오른쪽 그래프: Mean Spending & Purchase Count per User --------
# 왼쪽 y축 (막대)
ax3.bar(pub_day['weekday'], pub_day['user_amt'], color='pink')
ax3.set_ylabel('Mean Spending per User', color='pink')
ax3.tick_params(axis='y', labelcolor='pink')

# 오른쪽 y축 (선)
ax4 = ax3.twinx()
ax4.plot(pub_day['weekday'], pub_day['user_purchase'], color='blue', marker='s')
ax4.set_ylabel('Purchase Count per User', color='blue')
ax4.tick_params(axis='y', labelcolor='blue')

# 타이틀 및 x축
ax3.set_title('Mean Spending & Purchase Count per User - Public Sector')
ax3.set_xticks(range(len(pub_day['weekday'])))
ax3.set_xticklabels(pub_day['weekday'], rotation=45, ha='right')

# 저장 및 출력
plt.tight_layout()
plt.savefig('Public_Spending_Comparison_Weekly.png', bbox_inches='tight')
plt.show()

In [ ]:
# 요일 그룹 정의
weekend = ['Monday', 'Saturday', 'Sunday']
weekday = ['Tuesday', 'Wednesday', 'Thursday', 'Friday']

# 필터링 (직업명: Public Sector)
weekend_data = df[(df['job'] == 'public sector') & (df['weekday'].isin(weekend))]
weekday_data = df[(df['job'] == 'public sector') & (df['weekday'].isin(weekday))]

# category별 count
weekend_counts = weekend_data['category'].value_counts().reset_index()
weekend_counts.columns = ['category', 'weekend_count']

weekday_counts = weekday_data['category'].value_counts().reset_index()
weekday_counts.columns = ['category', 'weekday_count']

# 병합해서 category 순서 일치
merged = pd.merge(weekend_counts, weekday_counts, on='category', how='outer').fillna(0)
merged = merged.sort_values(by='weekend_count', ascending=False)

# 시각화
plt.figure(figsize=(12, 6))

# 막대 그래프: Mon/Sat/Sun
sns.barplot(data=merged, x='category', y='weekend_count', color='skyblue', label='Mon/Sat/Sun (Bar)')

# 선 그래프: Tue–Fri
plt.plot(merged['category'], merged['weekday_count'], color='orange', marker='o', label='Tue–Fri (Line)')

# 라벨 및 꾸미기
plt.title("Public Sector: Category Purchases – Mon/Sat/Sun(Bar) vs Tue–Fri(Line)", fontsize=14)
plt.xlabel("Category")
plt.ylabel("Purchase Count")
plt.xticks(rotation=45, ha='right')
plt.legend()
plt.tight_layout()
plt.savefig("Public_Sector_Category_Comparison_Line_Bar.png", bbox_inches='tight')
plt.show()


## 6. Regional Distribution

User population by US geographic belt.

In [ ]:
population = df.groupby('region_group')['user_id'].nunique()
# 색상 설정 (원하는 만큼 조정 가능)
colors = plt.get_cmap('Set3').colors  # 컬러맵에서 색상 추출

# 원형차트 그리기
plt.figure(figsize=(6, 6))
plt.pie(
    population,
    labels=population.index,
    autopct='%1.1f%%',
    startangle=90,
    colors=colors
)
plt.title('User Population by Region Group')
plt.axis('equal')  # 원형 유지
plt.tight_layout()
plt.savefig('User_Population_by_Region_Group.png', bbox_inches='tight')
plt.show()

---
## 📝 Key Findings

| # | Finding |
|---|---------|
| 1 | **Top 20 job groups** account for ~77% of transactions and ~78% of total spending |
| 2 | **Engineer** is the largest group by user count and shows a clear **Monday spending spike** |
| 3 | **Entertainment & food** categories dominate weekday purchases; **gas & grocery** rise on weekends |
| 4 | **Environment** job users have a lower per-user spend but higher purchase frequency |
| 5 | **Sun Belt & Middle Atlantic** regions contain the most users |

> ⚠️ *This dataset is synthetically generated for analytical practice.*
